In [1]:
import os
print(os.getcwd())
print(os.listdir())

/home/ubuntu/jupyter_notebooks/AXI_GA_20
['tasksets_500.json', '.ipynb_checkpoints', 'AXI_GA_NB_20.ipynb', 'design_AXI_GA_20.bit', 'design_AXI_GA_20.tcl', 'tasksets_300_8.json', 'design_AXI_GA_20.hwh', 'tasksets_300_9.json']


In [2]:
from pynq import Overlay
overlay = Overlay('design_AXI_GA_20.bit')

In [3]:
# If you also have the .hwh, PYNQ can often show the IP names:
print("IP dictionary keys:")
print(list(overlay.ip_dict.keys()))
print(overlay.clock_dict)

IP dictionary keys:
['AXI_GA_0', 'zynq_ultra_ps_e_0']
{0: {'enable': 1, 'divisor0': 50, 'divisor1': 1}, 1: {'enable': 1, 'divisor0': 50, 'divisor1': 2}, 2: {'enable': 0, 'divisor0': 4, 'divisor1': 1}, 3: {'enable': 0, 'divisor0': 4, 'divisor1': 1}}


In [4]:
from pynq import MMIO
import time
# Put your AXI base address here
GA_BASE  = 0xA0000000
GA_RANGE = 0x1000
mmio = MMIO(GA_BASE, GA_RANGE)

NTASKS = 64
NPROCS = 16

# ============================================================
# 2) Register map
# ============================================================
REG_CONTROL   = 0x00   # slv_reg0

EXEC_BASE     = 0x04   # slv_reg1
DEAD_BASE     = 0x44   # slv_reg17

BEST_TASK_BASE = 0x84  # slv_reg33
BEST_PROC_BASE = 0xB4  # slv_reg45
BEST_FIT       = 0xD4  # slv_reg53

def reg_offset(reg_index):
    return 4 * reg_index

In [5]:
# ------------------------------------------------------------------
# 1) Basic AXI sanity test: read/write a writable register
#    Use reg1, since reg33+ are output-driven by hardware
# ------------------------------------------------------------------
print("Initial reg1:", hex(mmio.read(EXEC_BASE)))
mmio.write(reg_offset(1), 0x1)
print("After write reg1:", hex(mmio.read(EXEC_BASE)))

Initial reg1: 0x0
After write reg1: 0x1


In [6]:
# ============================================================
# 3) Helpers
# ============================================================
def pack4_u8(vals):
    """
    Pack 4 unsigned 8-bit values into one 32-bit AXI register.
    Byte order matches:
      {task3, task2, task1, task0}
    so task0 goes in bits [7:0], task1 in [15:8], etc.
    """
    if len(vals) != 4:
        raise ValueError("pack4_u8 needs exactly 4 values")
    for v in vals:
        if not (0 <= int(v) <= 255):
            raise ValueError(f"value {v} out of 8-bit range")
    return (
        (int(vals[0]) & 0xFF) |
        ((int(vals[1]) & 0xFF) << 8) |
        ((int(vals[2]) & 0xFF) << 16) |
        ((int(vals[3]) & 0xFF) << 24)
    )

def unpack4_u8(word):
    return [
        word & 0xFF,
        (word >> 8) & 0xFF,
        (word >> 16) & 0xFF,
        (word >> 24) & 0xFF,
    ]

def unpack_tasks_6bit(words, ntasks=64):
    """
    slv_reg33..44 = 12 words = 384 bits = 64 tasks * 6 bits
    Reconstruct task IDs from LSB upward.
    """
    bitstream = 0
    for i, w in enumerate(words):
        bitstream |= (int(w) & 0xFFFFFFFF) << (32 * i)

    tasks = []
    for i in range(ntasks):
        tasks.append((bitstream >> (6 * i)) & 0x3F)
    return tasks

def unpack_procs_4bit(words, ntasks=64):
    """
    slv_reg45..52 = 8 words = 256 bits = 64 procs * 4 bits
    """
    bitstream = 0
    for i, w in enumerate(words):
        bitstream |= (int(w) & 0xFFFFFFFF) << (32 * i)

    procs = []
    for i in range(ntasks):
        procs.append((bitstream >> (4 * i)) & 0xF)
    return procs

def write_exec_deadline_arrays(exec_list, deadline_list, verify=True):
    if len(exec_list) != NTASKS:
        raise ValueError(f"Expected {NTASKS} exec values, got {len(exec_list)}")
    if len(deadline_list) != NTASKS:
        raise ValueError(f"Expected {NTASKS} deadline values, got {len(deadline_list)}")

    # Clear start first
    mmio.write(REG_CONTROL, 0)

    # Write execs into reg1..reg16
    for i in range(16):
        group = exec_list[4*i:4*i+4]
        word = pack4_u8(group)
        mmio.write(EXEC_BASE + 4*i, word)

    # Write deadlines into reg17..reg32
    for i in range(16):
        group = deadline_list[4*i:4*i+4]
        word = pack4_u8(group)
        mmio.write(DEAD_BASE + 4*i, word)

    if verify:
        read_exec = []
        for i in range(16):
            word = mmio.read(EXEC_BASE + 4*i)
            read_exec.extend(unpack4_u8(word))

        read_dead = []
        for i in range(16):
            word = mmio.read(DEAD_BASE + 4*i)
            read_dead.extend(unpack4_u8(word))

        ok_exec = (read_exec == [int(x) for x in exec_list])
        ok_dead = (read_dead == [int(x) for x in deadline_list])

        print("EXEC verify:", "PASS" if ok_exec else "FAIL")
        print("DEAD verify:", "PASS" if ok_dead else "FAIL")

        if not ok_exec:
            for i, (a, b) in enumerate(zip(exec_list, read_exec)):
                if int(a) != int(b):
                    print(f"First EXEC mismatch at task {i}: wrote {a}, read {b}")
                    break

        if not ok_dead:
            for i, (a, b) in enumerate(zip(deadline_list, read_dead)):
                if int(a) != int(b):
                    print(f"First DEAD mismatch at task {i}: wrote {a}, read {b}")
                    break

def read_back_input_regs():
    print("Execution registers:")
    for i in range(16):
        addr = EXEC_BASE + 4*i
        word = mmio.read(addr)
        print(f"reg{i+1:02d} @ 0x{addr:03X} = 0x{word:08X} -> {unpack4_u8(word)}")

    print("\nDeadline registers:")
    for i in range(16):
        addr = DEAD_BASE + 4*i
        word = mmio.read(addr)
        print(f"reg{i+17:02d} @ 0x{addr:03X} = 0x{word:08X} -> {unpack4_u8(word)}")

def start_accelerator():
    mmio.write(REG_CONTROL, 0x1)

def stop_accelerator():
    mmio.write(REG_CONTROL, 0x0)

def read_outputs():
    task_words = [mmio.read(BEST_TASK_BASE + 4*i) for i in range(12)]
    proc_words = [mmio.read(BEST_PROC_BASE + 4*i) for i in range(8)]
    fit_word   = mmio.read(BEST_FIT)

    tasks = unpack_tasks_6bit(task_words, NTASKS)
    procs = unpack_procs_4bit(proc_words, NTASKS)
    fit   = fit_word & 0x7FFFFF   # 23 bits

    return {
        "task_words": task_words,
        "proc_words": proc_words,
        "fit_word": fit_word,
        "best_task": tasks,
        "best_proc": procs,
        "best_fit": fit
    }

def print_outputs_summary(out, n=16):
    print("Best fit:", out["best_fit"])
    if (n!=0):
        print("\nFirst task output words:")
        for i, w in enumerate(out["task_words"]):
            print(f"reg{33+i:02d} = 0x{w:08X}")

        print("\nFirst proc output words:")
        for i, w in enumerate(out["proc_words"]):
            print(f"reg{45+i:02d} = 0x{w:08X}")

        print("\nFirst decoded genes:")
        for i in range(n):
            print(f"gene {i:02d}: task={out['best_task'][i]}, proc={out['best_proc'][i]}")

def print_schedule(schedule_eval, n=16):
    sched = schedule_eval["schedule"]
    print("gene | task | proc | exec | ddl | start | finish | miss")
    print("-" * 58)
    for r in sched[:n]:
        print(f"{r['gene']:>4} |"
              f"{r['task']:>5} |"
              f"{r['proc']:>5} |"
              f"{r['exec']:>5} |"
              f"{r['deadline']:>3} |"
              f"{r['start']:>5} |"
              f"{r['finish']:>6} |"
              f"{r['missed_deadline']:>4}")
        
def evaluate_schedule(best_task, best_proc, exec_list, deadline_list, nprocs=16, verbose=True):
    """
    Evaluate a decoded schedule from the accelerator.

    Inputs
    ------
    best_task     : list of 64 task IDs
    best_proc     : list of 64 processor IDs
    exec_list     : list of 64 execution times
    deadline_list : list of 64 deadlines
    nprocs        : number of processors

    Returns
    -------
    result : dict with:
        valid              -> overall validity flag
        errors             -> list of detected problems
        schedule           -> per-gene schedule info
        ART                -> average response time
        ATAT               -> average turnaround time
        DLMs               -> deadline misses
        schedule_length    -> makespan
    """

    ntasks = len(exec_list)
    errors = []

    # ------------------------------------------------------------
    # 1) Basic size checks
    # ------------------------------------------------------------
    if len(best_task) != ntasks:
        errors.append(f"best_task length {len(best_task)} does not match ntasks {ntasks}")
    if len(best_proc) != ntasks:
        errors.append(f"best_proc length {len(best_proc)} does not match ntasks {ntasks}")

    if errors:
        return {
            "valid": False,
            "errors": errors,
            "schedule": [],
            "ART": None,
            "ATAT": None,
            "DLMs": None,
            "schedule_length": None
        }

    # ------------------------------------------------------------
    # 2) Range checks
    # ------------------------------------------------------------
    for i, t in enumerate(best_task):
        if not (0 <= int(t) < ntasks):
            errors.append(f"gene {i}: task id {t} out of range [0, {ntasks-1}]")

    for i, p in enumerate(best_proc):
        if not (0 <= int(p) < nprocs):
            errors.append(f"gene {i}: proc id {p} out of range [0, {nprocs-1}]")

    # ------------------------------------------------------------
    # 3) Permutation check: every task must appear exactly once
    # ------------------------------------------------------------
    counts = [0] * ntasks
    for t in best_task:
        if 0 <= int(t) < ntasks:
            counts[int(t)] += 1

    missing_tasks = [i for i, c in enumerate(counts) if c == 0]
    repeated_tasks = [i for i, c in enumerate(counts) if c > 1]

    if missing_tasks:
        errors.append(f"Missing tasks: {missing_tasks}")
    if repeated_tasks:
        errors.append(f"Repeated tasks: {repeated_tasks}")

    # ------------------------------------------------------------
    # 4) Build actual schedule
    #    Tasks are processed in gene order.
    #    Each processor executes sequentially.
    #    Arrival time assumed 0.
    # ------------------------------------------------------------
    proc_time = [0] * nprocs
    results = []

    for gene_idx in range(ntasks):
        task_id = int(best_task[gene_idx])
        proc_id = int(best_proc[gene_idx])

        if not (0 <= task_id < ntasks) or not (0 <= proc_id < nprocs):
            continue

        start_time = proc_time[proc_id]
        finish_time = start_time + int(exec_list[task_id])

        results.append({
            "gene": gene_idx,
            "task": task_id,
            "proc": proc_id,
            "exec": int(exec_list[task_id]),
            "deadline": int(deadline_list[task_id]),
            "start": start_time,
            "finish": finish_time,
            "missed_deadline": int(finish_time > int(deadline_list[task_id]))
        })

        proc_time[proc_id] = finish_time

    # ------------------------------------------------------------
    # 5) Metrics
    # ------------------------------------------------------------
    if len(results) == ntasks and not any("out of range" in e for e in errors):
        response_times = [r["start"] for r in results]
        turnaround_times = [r["finish"] for r in results]
        dlms = sum(r["missed_deadline"] for r in results)
        art = sum(response_times) / len(response_times)
        atat = sum(turnaround_times) / len(turnaround_times)
        schedule_length = max((r["finish"] for r in results), default=0)
    else:
        art = None
        atat = None
        dlms = None
        schedule_length = None

    valid = len(errors) == 0

    if verbose:
        print("Schedule valid:", valid)
        if errors:
            print("\nErrors found:")
            for e in errors:
                print(" -", e)

        if valid:
            print("\nMetrics:")
            print("ART            =", art)
            print("ATAT           =", atat)
            print("DLMs           =", dlms)
            print("Schedule length=", schedule_length)

    return {
        "valid": valid,
        "errors": errors,
        "schedule": results,
        "ART": art,
        "ATAT": atat,
        "DLMs": dlms,
        "schedule_length": schedule_length
    }

def run_benchmark_accelerator(
    dataset,
    bench_idx=6,
    sleep_s=0.01,
    verify_inputs=True,
    stop_after_run=True,
    print_summary=True,
    summary_n=0,
    verbose_eval=True,
    nprocs=16
):
    # ------------------------------------------------------------
    # 1) Select benchmark and extract exec/deadline arrays
    # ------------------------------------------------------------
    bench = dataset[bench_idx]

    exec_list = [int(t["exec"]) for t in bench["tasks"]]
    deadline_list = [int(t["deadline"]) for t in bench["tasks"]]

    print(f"Benchmark: {bench['benchmark']}")

    # ------------------------------------------------------------
    # 2) Write input registers
    # ------------------------------------------------------------
    write_exec_deadline_arrays(exec_list, deadline_list, verify=verify_inputs)

    # ------------------------------------------------------------
    # 3) Start accelerator
    # ------------------------------------------------------------
    start_accelerator()
    time.sleep(sleep_s)

    if stop_after_run:
        stop_accelerator()

    # ------------------------------------------------------------
    # 4) Read outputs
    # ------------------------------------------------------------
    ctrl_reg = mmio.read(reg_offset(REG_CONTROL))
    print("Control Reg:", hex(ctrl_reg))

    out = read_outputs()

    if print_summary:
        print_outputs_summary(out, n=summary_n)

    # ------------------------------------------------------------
    # 5) Evaluate schedule
    # ------------------------------------------------------------
    sched_eval = evaluate_schedule(
        best_task=out["best_task"],
        best_proc=out["best_proc"],
        exec_list=exec_list,
        deadline_list=deadline_list,
        nprocs=nprocs,
        verbose=verbose_eval
    )

    # Extract metrics directly
    art = sched_eval["ART"]
    atat = sched_eval["ATAT"]
    dlms = sched_eval["DLMs"]

    # ------------------------------------------------------------
    # 6) Return everything useful
    # ------------------------------------------------------------
    return {
        "benchmark_info": bench,
        "bench_idx": bench_idx,
        "exec_list": exec_list,
        "deadline_list": deadline_list,
        "outputs": out,
        "schedule_eval": sched_eval,
        "control_reg": ctrl_reg,
        "art": art,
        "atat": atat,
        "dlms": dlms
    }

In [53]:
# ============================================================
# 4) Load JSON dataset
# ============================================================
import json
# json_file = "tasksets_500.json"   # uploaded file name
# json_file = "tasksets_300_8.json"   # uploaded file name
json_file = "tasksets_300_9.json"   # uploaded file name


with open(json_file, "r") as f:
    dataset = json.load(f)

print("Benchmarks in file:", len(dataset))

Benchmarks in file: 300


In [75]:
import numpy as np

# Extract metrics
art_vals  = np.array([r["art"]  for r in results if r["art"] is not None])
atat_vals = np.array([r["atat"] for r in results if r["atat"] is not None])
dlm_vals  = np.array([r["dlms"] for r in results if r["dlms"] is not None])

# Compute statistics
stats = {
    "ART": {
        "mean": np.mean(art_vals),
        "std":  np.std(art_vals),
        "var":  np.var(art_vals)
    },
    "ATAT": {
        "mean": np.mean(atat_vals),
        "std":  np.std(atat_vals),
        "var":  np.var(atat_vals)
    },
    "DLM": {
        "mean": np.mean(dlm_vals),
        "std":  np.std(dlm_vals),
        "var":  np.var(dlm_vals)
    }
}

# Print nicely
for k, v in stats.items():
    print(f"\n{k}:")
    print(f"  Mean = {v['mean']:.4f}")
    print(f"  Std  = {v['std']:.4f}")
    print(f"  Var  = {v['var']:.4f}")


ART:
  Mean = 9.3169
  Std  = 1.5213
  Var  = 2.3145

ATAT:
  Mean = 20.5043
  Std  = 2.3195
  Var  = 5.3802

DLM:
  Mean = 1.1500
  Std  = 0.8986
  Var  = 0.8075


In [74]:
results = []

for bench_idx in range(0, 300):
    print("\n" + "="*60)
    print(f"Running benchmark index {bench_idx}")
    print("="*60)

    result = run_benchmark_accelerator(dataset, bench_idx=bench_idx, sleep_s=0.001, summary_n=0)
    results.append(result)


Running benchmark index 0
Benchmark: 0
EXEC verify: PASS
DEAD verify: PASS
Control Reg: 0x0
Best fit: 3673643
Schedule valid: True

Metrics:
ART            = 7.984375
ATAT           = 20.421875
DLMs           = 1
Schedule length= 92

Running benchmark index 1
Benchmark: 1
EXEC verify: PASS
DEAD verify: PASS
Control Reg: 0x0
Best fit: 3673647
Schedule valid: True

Metrics:
ART            = 7.0
ATAT           = 16.5625
DLMs           = 1
Schedule length= 132

Running benchmark index 2
Benchmark: 2
EXEC verify: PASS
DEAD verify: PASS
Control Reg: 0x0
Best fit: 3673646
Schedule valid: True

Metrics:
ART            = 7.953125
ATAT           = 17.609375
DLMs           = 1
Schedule length= 135

Running benchmark index 3
Benchmark: 3
EXEC verify: PASS
DEAD verify: PASS
Control Reg: 0x0
Best fit: 3673449
Schedule valid: True

Metrics:
ART            = 10.984375
ATAT           = 22.703125
DLMs           = 1
Schedule length= 111

Running benchmark index 4
Benchmark: 4
EXEC verify: PASS
DEAD veri


ART:
  Mean = 5.5438
  Std  = 1.1162
  Var  = 1.2458

ATAT:
  Mean = 14.3875
  Std  = 2.0940
  Var  = 4.3848

DLM:
  Mean = 0.3000
  Std  = 0.4583
  Var  = 0.2100
